# Part 4 — Backtesting y Comparación de Estrategias

**Proyecto:** Optimización de Portafolios con Métodos Clásicos, Machine Learning y Deep Learning
**Autor:** Jacqueline Alessandra Espinosa Celestino
**Matrícula:** 1953485
**Institución:** FCFM — UANL
**Fecha:** Octubre 2026

---

En las partes anteriores construí portafolios optimizados con tres enfoques: Markowitz clásico, Machine Learning (Random Forest y XGBoost) y Deep Learning (LSTM). Cada notebook evaluó su estrategia por separado, pero eso no permite una comparación justa porque los periodos exactos de evaluación y la forma de medir difieren.

En esta parte final, pongo todas las estrategias en las mismas condiciones: mismo periodo de backtest (2024), mismo rebalanceo mensual, mismas métricas. Así puedo concluir cuál enfoque fue mejor — y por qué.

## Marco teórico

### Backtesting

Un backtest simula cómo habría funcionado una estrategia si la hubiéramos aplicado sobre datos reales del pasado. La clave es respetar la línea temporal: los modelos solo pueden usar datos disponibles *antes* de cada decisión. Violar esto se llama look-ahead bias y invalida cualquier resultado. Por eso entrené todos los modelos con datos de 2020–2023 y el backtest corre sobre 2024.

### Rebalanceo mensual

En la práctica nadie rebalancea a diario — los costos de transacción lo harían inviable. El rebalanceo mensual es un estándar que balancea:
- Adaptarse a nueva información de los modelos
- Mantener costos operativos razonables

Cada primer día hábil del mes:
1. Los modelos generan predicciones con features disponibles hasta ese día
2. Se optimizan los pesos del portafolio
3. Los pesos se mantienen fijos hasta el siguiente rebalanceo

### Métricas de evaluación

**Retorno anualizado:** $(1 + R_{total})^{252/n} - 1$

**Volatilidad anualizada:** $\sigma_{diaria} \times \sqrt{252}$

**Sharpe Ratio:** $SR = \frac{R_p - R_f}{\sigma_p}$ con $R_f = 5\%$

**Sortino Ratio:** Igual que Sharpe pero penaliza solo volatilidad negativa: $\frac{R_p - R_f}{\sigma_{down}}$

**Max Drawdown:** La peor caída desde un máximo: $MDD = \min_t \left(\frac{V_t}{\max_{s \leq t} V_s} - 1\right)$

**Calmar Ratio:** $\frac{R_{anual}}{|MDD|}$ — combina retorno y peor escenario.

**Turnover:** Cambio promedio en pesos por rebalanceo. Turnover alto = más transacciones.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

TICKERS   = ['AAPL','MSFT','JPM','JNJ','XOM','KO','AMZN','NVDA','CAT','PG']
BENCHMARK = 'SPY'
START     = '2020-01-01'
END       = '2024-12-31'
RF_RATE   = 0.05
N         = len(TICKERS)
SPLIT     = '2024-01-01'

print(f"Activos: {TICKERS}")
print(f"Periodo total: {START} — {END}")
print(f"Entrenamiento: {START} — 2023-12-31")
print(f"Backtest: {SPLIT} — {END}")

## Descarga de datos

Mismos 10 activos y benchmark que en las partes anteriores.

In [ ]:
all_tickers = TICKERS + [BENCHMARK]
data = yf.download(all_tickers, start=START, end=END, auto_adjust=True)

if isinstance(data.columns, pd.MultiIndex):
    prices = data['Close'][all_tickers]
else:
    prices = data[all_tickers]

failed = prices.columns[prices.isna().all()]
if len(failed) > 0:
    print(f"No se pudieron descargar: {list(failed)}")
prices = prices.dropna()

print(f"Datos: {prices.shape[0]} días, {prices.shape[1]} activos")
print(f"Rango: {prices.index[0].strftime('%Y-%m-%d')} — {prices.index[-1].strftime('%Y-%m-%d')}")

## Ingeniería de features

Las mismas 12 features técnicas de Parts 2 y 3. Target: retorno a 21 días.

In [ ]:
FEATURE_COLS = [
    'ret_1d','ret_5d','ret_10d','ret_21d',
    'sma10_ratio','sma20_ratio','sma50_ratio',
    'rsi_14','macd_diff','bb_pct',
    'vol_10d','vol_21d',
]

def compute_features(close):
    df = pd.DataFrame(index=close.index)
    r1 = close.pct_change()
    df['ret_1d']  = r1
    df['ret_5d']  = close.pct_change(5)
    df['ret_10d'] = close.pct_change(10)
    df['ret_21d'] = close.pct_change(21)

    df['sma10_ratio'] = close / close.rolling(10).mean()
    df['sma20_ratio'] = close / close.rolling(20).mean()
    df['sma50_ratio'] = close / close.rolling(50).mean()

    gain = r1.clip(lower=0).rolling(14).mean()
    loss = (-r1.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['rsi_14'] = 100 - 100 / (1 + rs)

    ema12 = close.ewm(span=12).mean()
    ema26 = close.ewm(span=26).mean()
    macd  = ema12 - ema26
    df['macd_diff'] = macd - macd.ewm(span=9).mean()

    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    df['bb_pct'] = (close - (bb_mid - 2*bb_std)) / (4*bb_std)

    df['vol_10d'] = r1.rolling(10).std()
    df['vol_21d'] = r1.rolling(21).std()
    return df

stock_data = {}
for t in TICKERS:
    feats = compute_features(prices[t])
    feats['target'] = prices[t].pct_change(21).shift(-21)
    feats = feats.dropna()
    stock_data[t] = feats

print(f"Features para {len(stock_data)} activos")

In [ ]:
train_data = {}
test_data  = {}
for t in TICKERS:
    df = stock_data[t]
    m  = df.index < pd.Timestamp(SPLIT)
    train_data[t] = df[m]
    test_data[t]  = df[~m]

ex = list(train_data.keys())[0]
print(f"Train: {train_data[ex].shape[0]} obs/acción  |  Test: {test_data[ex].shape[0]} obs/acción")

## Entrenamiento de modelos

### Random Forest y XGBoost

Mismos hiperparámetros que en Part 2.

In [ ]:
rf_models  = {}
xgb_models = {}

for t in TICKERS:
    X_tr = train_data[t][FEATURE_COLS].values
    y_tr = train_data[t]['target'].values

    rf = RandomForestRegressor(n_estimators=200, max_depth=6,
                               min_samples_leaf=20, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    rf_models[t] = rf

    xgb = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8,
                       reg_alpha=0.1, reg_lambda=1.0,
                       random_state=42, verbosity=0)
    xgb.fit(X_tr, y_tr)
    xgb_models[t] = xgb

print("RF y XGBoost entrenados para las 10 acciones")

### LSTM

Misma arquitectura de Part 3 (2 capas LSTM, 64 + 32 unidades). Reduzco épocas máximas a 30 porque aquí el foco es la comparación, no exprimir el modelo.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEQ_LEN = 60
scalers     = {}
lstm_models = {}

for t in TICKERS:
    df    = stock_data[t]
    dates = df.index
    X_all = df[FEATURE_COLS].values
    y_all = df['target'].values

    tr_mask = dates < pd.Timestamp(SPLIT)
    sc = StandardScaler()
    sc.fit(X_all[tr_mask])
    X_sc = sc.transform(X_all)
    scalers[t] = sc

    X_seq, y_seq, d_seq = [], [], []
    for i in range(SEQ_LEN, len(X_sc)):
        X_seq.append(X_sc[i-SEQ_LEN:i])
        y_seq.append(y_all[i])
        d_seq.append(dates[i])

    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)
    d_seq = np.array(d_seq)

    split_mask = np.array([d < pd.Timestamp(SPLIT) for d in d_seq])
    X_tr_s = X_seq[split_mask]
    y_tr_s = y_seq[split_mask]

    tf.random.set_seed(42)
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, len(FEATURE_COLS))),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    es = EarlyStopping(monitor='val_loss', patience=10,
                       restore_best_weights=True, verbose=0)
    model.fit(X_tr_s, y_tr_s, epochs=30, batch_size=32,
              validation_split=0.15, callbacks=[es], verbose=0)
    lstm_models[t] = model

print(f"LSTM entrenados: {len(lstm_models)} acciones")

## Funciones de optimización y predicción

In [ ]:
def optimize_sharpe(mu, cov, rf=RF_RATE):
    """Portafolio de maximo Sharpe. Pesos entre 0 y 1, suman 1."""
    n = len(mu)
    def neg_sr(w):
        ret = w @ mu
        vol = np.sqrt(w @ cov @ w)
        return -(ret - rf) / vol if vol > 1e-10 else 10.0
    res = minimize(neg_sr, np.ones(n)/n, method='SLSQP',
                   bounds=[(0,1)]*n,
                   constraints={'type':'eq','fun': lambda w: w.sum()-1})
    return res.x if res.success else np.ones(n)/n

# Covarianza anualizada del entrenamiento
train_rets = pd.DataFrame({t: train_data[t]['ret_1d'] for t in TICKERS})
cov_annual = train_rets.cov().values * 252

def predict_mu(date, strategy):
    """Retornos esperados anualizados segun la estrategia."""
    mu = np.zeros(N)
    if strategy == 'historico':
        for i, t in enumerate(TICKERS):
            mu[i] = train_data[t]['ret_1d'].mean() * 252
    elif strategy == 'rf':
        for i, t in enumerate(TICKERS):
            avail = stock_data[t].loc[stock_data[t].index <= pd.Timestamp(date), FEATURE_COLS]
            if len(avail) == 0: continue
            mu[i] = rf_models[t].predict(avail.iloc[-1:].values)[0] * (252/21)
    elif strategy == 'xgb':
        for i, t in enumerate(TICKERS):
            avail = stock_data[t].loc[stock_data[t].index <= pd.Timestamp(date), FEATURE_COLS]
            if len(avail) == 0: continue
            mu[i] = xgb_models[t].predict(avail.iloc[-1:].values)[0] * (252/21)
    elif strategy == 'lstm':
        for i, t in enumerate(TICKERS):
            avail = stock_data[t].loc[stock_data[t].index <= pd.Timestamp(date), FEATURE_COLS]
            if len(avail) < SEQ_LEN: continue
            X_raw = avail.iloc[-SEQ_LEN:].values
            X_in  = scalers[t].transform(X_raw).reshape(1, SEQ_LEN, len(FEATURE_COLS))
            mu[i] = lstm_models[t].predict(X_in, verbose=0)[0,0] * (252/21)
    return mu

print("Funciones listas")

## Backtest con rebalanceo mensual

El motor funciona así:
1. Identifico el primer día hábil de cada mes en 2024
2. En esa fecha, cada estrategia calcula sus pesos
3. Los pesos se mantienen fijos durante el mes
4. Los retornos diarios reales del portafolio se acumulan

Las estrategias:
- **Equal Weight (1/N):** pesos iguales siempre
- **Histórico:** retornos medios históricos → Markowitz
- **Random Forest:** predicciones RF → Markowitz
- **XGBoost:** predicciones XGB → Markowitz
- **LSTM:** predicciones LSTM → Markowitz
- **SPY:** benchmark pasivo

In [ ]:
# Retornos diarios del periodo de backtest
bt_prices  = prices.loc[prices.index >= pd.Timestamp(SPLIT)]
bt_returns = bt_prices[TICKERS].pct_change().dropna()
spy_rets   = bt_prices[BENCHMARK].pct_change().dropna()

# Asegurar que comparten indice
common_idx = bt_returns.index.intersection(spy_rets.index)
bt_returns = bt_returns.loc[common_idx]
spy_rets   = spy_rets.loc[common_idx]

# Fechas de rebalanceo: primer dia habil de cada mes
rebal_dates = (bt_returns.groupby(bt_returns.index.to_period('M'))
               .apply(lambda x: x.index[0]).values)

print(f"Backtest: {bt_returns.index[0].strftime('%Y-%m-%d')} — {bt_returns.index[-1].strftime('%Y-%m-%d')}")
print(f"Dias de trading: {len(bt_returns)}")
print(f"Meses de rebalanceo: {len(rebal_dates)}")

In [ ]:
STRATEGIES = ['equal', 'historico', 'rf', 'xgb', 'lstm']
LABELS = {
    'equal':'Equal Weight', 'historico':'Historico',
    'rf':'Random Forest', 'xgb':'XGBoost', 'lstm':'LSTM', 'spy':'SPY'
}

port_daily = {}          # retornos diarios de cada portafolio
weight_hist = {}         # historial de pesos para turnover

for s in STRATEGIES:
    print(f"Backtesting {LABELS[s]}...")
    daily_rets = pd.Series(0.0, index=bt_returns.index)
    w_hist = []

    for k, rd in enumerate(rebal_dates):
        rd_ts = pd.Timestamp(rd)
        # Pesos
        if s == 'equal':
            w = np.ones(N) / N
        else:
            mu = predict_mu(rd, s)
            w  = optimize_sharpe(mu, cov_annual)

        w_hist.append({'date': rd_ts, 'w': w.copy()})

        # Periodo de vigencia de estos pesos
        if k < len(rebal_dates) - 1:
            end_ts = pd.Timestamp(rebal_dates[k+1])
            mask = (bt_returns.index >= rd_ts) & (bt_returns.index < end_ts)
        else:
            mask = bt_returns.index >= rd_ts

        daily_rets[mask] = bt_returns.loc[mask].values @ w

    port_daily[s] = daily_rets
    weight_hist[s] = w_hist

port_daily['spy'] = spy_rets

# Valor acumulado (base $1)
cum = pd.DataFrame({s: (1 + port_daily[s]).cumprod() for s in port_daily})

print(f"\nBacktest completado. Rango de valores finales:")
for s in cum.columns:
    print(f"  {LABELS[s]:20s}  ${cum[s].iloc[-1]:.4f}")

## Retornos acumulados

La grafica central del backtest. Muestra como habria crecido $1 invertido en cada estrategia durante 2024.

In [ ]:
COLORS = {
    'equal':'#888888', 'historico':'#2196F3', 'rf':'#4CAF50',
    'xgb':'#FF9800', 'lstm':'#9C27B0', 'spy':'#000000'
}

fig, ax = plt.subplots(figsize=(12, 6))
for s in ['spy','equal','historico','rf','xgb','lstm']:
    ls = '--' if s == 'spy' else '-'
    lw = 1.5 if s == 'spy' else 2
    ax.plot(cum.index, cum[s], label=LABELS[s],
            color=COLORS[s], linestyle=ls, linewidth=lw)

ax.set_title('Backtest 2024: Retornos Acumulados', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Valor del portafolio (base $1)')
ax.legend(loc='upper left', framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.axhline(1, color='gray', ls=':', alpha=0.5)
plt.tight_layout()
plt.show()

## Drawdown

El drawdown mide la caida desde el maximo historico del portafolio. Es lo que mas duele al inversionista — nadie disfruta ver su cuenta caer un 10% aunque al final del año cierre positivo.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for s in ['spy','equal','historico','rf','xgb','lstm']:
    vals = cum[s].values
    run_max = np.maximum.accumulate(vals)
    dd = (vals / run_max - 1) * 100
    ls = '--' if s == 'spy' else '-'
    ax.plot(cum.index, dd, label=LABELS[s], color=COLORS[s],
            linestyle=ls, linewidth=1.5, alpha=0.85)

ax.set_title('Drawdown por Estrategia (2024)', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Drawdown (%)')
ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', ls='-', alpha=0.3)
plt.tight_layout()
plt.show()

## Sharpe Ratio rolling (63 dias)

El Sharpe no es constante. Una ventana de 63 dias (~3 meses) muestra como cambia a lo largo del año segun las condiciones del mercado.

In [ ]:
W = 63
rf_d = RF_RATE / 252

fig, ax = plt.subplots(figsize=(12, 5))

for s in ['spy','equal','historico','rf','xgb','lstm']:
    r = port_daily[s].values
    roll_sr = pd.Series(r).rolling(W).apply(
        lambda x: (x.mean() - rf_d) / x.std() * np.sqrt(252) if x.std() > 0 else 0,
        raw=True
    )
    ls = '--' if s == 'spy' else '-'
    ax.plot(cum.index, roll_sr.values, label=LABELS[s], color=COLORS[s],
            linestyle=ls, linewidth=1.5, alpha=0.85)

ax.set_title('Sharpe Ratio Rolling (63 dias)', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Sharpe Ratio')
ax.legend(loc='best', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', ls=':', alpha=0.5)
plt.tight_layout()
plt.show()

## Evolucion de pesos

Graficas de areas apiladas que muestran como cada estrategia asigna capital mes a mes. Equal Weight es constante por definicion; las demas reflejan las señales de sus modelos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for ax, s in zip(axes.flat, ['historico','rf','xgb','lstm']):
    wh = weight_hist[s]
    months = [w['date'].strftime('%b') for w in wh]
    wmat   = np.array([w['w'] for w in wh])   # (meses, N)

    ax.stackplot(range(len(months)), wmat.T, labels=TICKERS, alpha=0.8)
    ax.set_title(LABELS[s], fontsize=12)
    ax.set_ylabel('Peso')
    ax.set_ylim(0, 1)
    ax.set_xticks(range(len(months)))
    ax.set_xticklabels(months, fontsize=8)

handles, lbls = axes[0,0].get_legend_handles_labels()
fig.legend(handles, lbls, loc='upper center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, 1.04))
fig.suptitle('Asignacion de Pesos por Estrategia (2024)', fontsize=14, y=1.08)
plt.tight_layout()
plt.subplots_adjust(hspace=0.35)
plt.show()

## Tabla de metricas

La comparacion final. Aqui se ve cual estrategia fue mejor y en que dimension.

In [ ]:
def calc_metrics(daily_rets, label):
    r   = daily_rets.values
    n   = len(r)
    cum_r  = (1 + pd.Series(r)).cumprod()
    total  = cum_r.iloc[-1] - 1
    ann_r  = (1 + total) ** (252 / n) - 1
    ann_v  = r.std() * np.sqrt(252)
    sr     = (r.mean() - RF_RATE/252) / r.std() * np.sqrt(252) if r.std()>0 else 0

    down   = r[r < 0]
    ds     = down.std() * np.sqrt(252) if len(down)>0 else 1e-6
    sortino = (ann_r - RF_RATE) / ds

    run_max = np.maximum.accumulate(cum_r.values)
    mdd     = (cum_r.values / run_max - 1).min()
    calmar  = ann_r / abs(mdd) if abs(mdd) > 0 else 0

    return {
        'Estrategia':     label,
        'Ret Total (%)':  round(total*100, 2),
        'Ret Anual (%)':  round(ann_r*100, 2),
        'Vol (%)':        round(ann_v*100, 2),
        'Sharpe':         round(sr, 3),
        'Sortino':        round(sortino, 3),
        'Max DD (%)':     round(mdd*100, 2),
        'Calmar':         round(calmar, 3),
    }

rows = [calc_metrics(port_daily[s], LABELS[s])
        for s in ['equal','historico','rf','xgb','lstm','spy']]
metrics = pd.DataFrame(rows).set_index('Estrategia')

print("=" * 80)
print("RESUMEN DEL BACKTEST 2024")
print("=" * 80)
print()
print(metrics.to_string())
print()

best_sharpe = metrics['Sharpe'].idxmax()
best_ret    = metrics['Ret Anual (%)'].idxmax()
least_dd    = metrics['Max DD (%)'].idxmax()   # menos negativo = mejor

print(f"Mejor Sharpe:        {best_sharpe} ({metrics.loc[best_sharpe,'Sharpe']:.3f})")
print(f"Mayor retorno:       {best_ret} ({metrics.loc[best_ret,'Ret Anual (%)']:.2f}%)")
print(f"Menor drawdown:      {least_dd} ({metrics.loc[least_dd,'Max DD (%)']:.2f}%)")

## Turnover

Cuanto cambian los pesos en cada rebalanceo. Alto turnover implica mas transacciones y costos.

In [ ]:
print("=== Turnover promedio por estrategia ===\n")

turnover_data = {}
for s in ['historico','rf','xgb','lstm']:
    wh = weight_hist[s]
    to_list = []
    for i in range(1, len(wh)):
        to = np.sum(np.abs(wh[i]['w'] - wh[i-1]['w'])) / 2
        to_list.append(to)
    avg = np.mean(to_list) if to_list else 0
    turnover_data[s] = avg
    print(f"  {LABELS[s]:20s}  {avg:.4f}  ({avg*100:.2f}% del portafolio)")

print(f"  {'Equal Weight':20s}  0.0000  (0.00%)")
print()
print("Turnover = sum(|cambio en peso|) / 2 por rebalanceo.")
print("Un valor de 0.10 significa que ~10% del capital se reasigna cada mes.")

## Comparacion visual

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
cols   = ['Ret Anual (%)','Sharpe','Max DD (%)','Sortino']
titles = ['Retorno Anualizado (%)','Sharpe Ratio','Max Drawdown (%)','Sortino Ratio']
bar_c  = ['#888888','#2196F3','#4CAF50','#FF9800','#9C27B0','#000000']

for ax, col, title in zip(axes, cols, titles):
    vals = metrics[col].values
    bars = ax.barh(metrics.index, vals, color=bar_c)
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3, axis='x')

    # Ampliar limites para que quepan las etiquetas
    vmin, vmax = vals.min(), vals.max()
    margin = (vmax - vmin) * 0.25 if vmax != vmin else abs(vmax) * 0.3
    if vmin >= 0:
        ax.set_xlim(0, vmax + margin)
    elif vmax <= 0:
        ax.set_xlim(vmin - margin, 0)
    else:
        ax.set_xlim(vmin - margin, vmax + margin)

    for b, v in zip(bars, vals):
        ha = 'left' if v >= 0 else 'right'
        ax.text(b.get_width(), b.get_y()+b.get_height()/2,
                f' {v:.2f}', va='center', ha=ha, fontsize=8)

plt.suptitle('Metricas Clave por Estrategia', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Retornos mensuales por estrategia

Una vista mes a mes permite ver en cuales meses cada estrategia gano o perdio.

In [ ]:
monthly = pd.DataFrame()
for s in list(STRATEGIES) + ['spy']:
    monthly[LABELS[s]] = port_daily[s].resample('ME').apply(
        lambda x: (1 + x).prod() - 1
    ) * 100

print("Retornos mensuales (%):")
print(monthly.round(2).to_string())
print()

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(monthly))
width = 0.13
strats_plot = list(LABELS.values())

for i, col in enumerate(monthly.columns):
    offset = (i - len(monthly.columns)/2) * width
    ax.bar(x + offset, monthly[col], width, label=col, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(monthly.index.strftime('%b %Y'), fontsize=9)
ax.set_ylabel('Retorno (%)')
ax.set_title('Retornos Mensuales por Estrategia', fontsize=14)
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='gray', lw=0.8)
plt.tight_layout()
plt.show()

## Conclusiones

Despues de comparar las cinco estrategias en igualdad de condiciones durante 2024, hay varios puntos que me parecen importantes.

**Modelos ML vs. enfoque clasico**

Los modelos de Machine Learning generaron asignaciones diferentes a las del Markowitz clasico. La tabla de metricas muestra que las diferencias en retorno no siempre van acompañadas de mejor Sharpe — un portafolio con mayor retorno pero mucho mas volatil no es necesariamente superior. El Sharpe Ratio captura ese balance, y ahi se ve cual enfoque realmente agrego valor ajustado por riesgo.

**El LSTM en contexto**

Como vimos en Part 3, las predicciones del LSTM tuvieron R² negativo para la mayoria de las acciones. Sin embargo, al combinarse en un portafolio con optimizacion de Markowitz, esas señales debiles pueden traducirse en asignaciones que capturan dinamicas temporales que los modelos de arbol no ven. Los resultados del backtest muestran si esa ventaja teorica se materializo en 2024.

**Turnover y costos**

El analisis de turnover revela que las estrategias basadas en modelos cambian mas sus asignaciones que el enfoque historico. En un contexto real, habria que restar comisiones (tipicamente 5–10 bps por operacion), spread bid-ask e impacto de mercado. Un modelo que genera 15% de retorno pero con 50% de turnover mensual podria ser peor que uno con 12% y turnover de 5%.

**Limitaciones**

1. Backtest de solo un año — los resultados podrian no generalizarse
2. Sin costos de transaccion explicitos
3. Covarianza estatica (del entrenamiento), no dinamica
4. Modelos entrenados una sola vez, sin re-entrenamiento mensual
5. Sin restricciones de liquidez o concentracion sectorial

**Posibles extensiones**

Para mejorar el analisis se podrian implementar: re-entrenamiento mensual (expanding window), costos de transaccion explicitos, modelos de covarianza dinamica (GARCH), restricciones de concentracion (max 20% por activo), y optimizacion walk-forward.

---

*Este notebook cierra el proyecto. La comparacion muestra que no hay una estrategia universalmente superior — la eleccion depende del horizonte de inversion, la tolerancia al riesgo y los costos operativos.*